# Physical Activity
**Table of contents**

- Overview

- Setup

  - Authentication Token

- Query

  - Output Description

- Related Links

## Overview

Use the Physical Activity API to calculate emissions from physical activities and operations. The API supports emissions accounting aligned with Scope 3 Category 15 (Investments) under the GHG Protocol, enabling organizations to estimate financed emissions based on physical activity data such as area-based metrics for commercial real estate and other physical operations.

In the API request, you provide the activity type, value, and unit of measurement. The API response provides a breakdown of emissions by gas and an aggregate CO2e value. The API supports attribution for private companies using equity/debt based calculations or EVIC (Enterprise Value Including Cash).

## Setup
Ensure that Python 3+ is installed on your system.

<b>Note:</b> To run this notebook, you must first add your credentials to `'../../../auth/secrets.ini'` in the following format:

```
[EAPI]
api.pat_token = <Your Emissions API PAT token>
api.tenant_id = <Your Emissions API Tenant Id>
```

In [ ]:
# Install the prerequisite Python packages.

%pip install pandas configparser IPython requests

In [ ]:
import pandas as pd
import configparser
import requests
import json
from IPython.display import display as display_summary

### Authentication Token
Run the following code snippet to generate a Bearer Token by using your api_key configured in secrets.ini.

In [ ]:
config = configparser.RawConfigParser()
config.read(['../../../auth/secrets.ini','../../../auth/config.ini'])

EAPI_PAT_TOKEN      = config.get('EAPI', 'api.pat_token')
EAPI_TENANT_ID      = config.get('EAPI', 'api.tenant_id')
EAPI_CLIENT_ID      = 'ghgemissions-' + EAPI_TENANT_ID

EAPI_AUTH_CLIENT_ID = 'saascore-' + EAPI_TENANT_ID
EAPI_AUTH_ENDPOINT  = config.get('EAPI', 'api.auth_endpoint')

EAPI_BASE_URL       = config.get('EAPI', 'api.base_url')
EAPI_ENDPOINT       = f"{EAPI_BASE_URL}/physical-activity"

auth_request_headers: dict = {}
auth_request_headers["X-IBM-Client-Id"] = EAPI_AUTH_CLIENT_ID
auth_request_headers["X-IBM-Envizi-Pat"] = EAPI_PAT_TOKEN

verify = True

auth_url = f"{EAPI_AUTH_ENDPOINT}"

response = requests.post(url = auth_url,
                        headers = auth_request_headers,
                        verify  = verify
                       )
if response.status_code == 200:
    jwt_token = response.text
    print("Authentication Success")
else:     
    print("Authentication Failed")
    print(response.text)

## Query

The example request payload (below) queries IBM Envizi - Emissions API for the emissions generated from 0.1 square kilometers of commercial real estate in the USA:

In [ ]:
payload = {
  "activity": {
    "type": "commercial real estate",
    "value": 0.1,
    "unit": "km2"
  },
  "location": {
    "country": "USA"
  },
  "time": {
    "date": "2025-01-23"
  }
}

## Query with Attribution (Equity/Debt Based)

The example request payload (below) includes attribution for private companies using equity and debt values. This is useful for calculating financed emissions based on ownership structure:

In [ ]:
payload_with_equity_debt = {
  "activity": {
    "type": "commercial real estate",
    "value": 0.1,
    "unit": "km2"
  },
  "location": {
    "country": "USA"
  },
  "time": {
    "date": "2025-01-23"
  },
  "attribution": {
    "outstandingAmount": 1000000,
    "totalEquity": 3000000,
    "totalDebt": 2000000
  }
}

## Query with Attribution (EVIC Based)

The example request payload (below) includes attribution using EVIC (Enterprise Value Including Cash). This method is useful for calculating financed emissions based on enterprise value:

In [ ]:
payload_with_evic = {
  "activity": {
    "type": "commercial real estate",
    "value": 0.1,
    "unit": "km2"
  },
  "location": {
    "country": "USA"
  },
  "time": {
    "date": "2025-01-23"
  },
  "attribution": {
    "outstandingAmount": 1000000.0,
    "evic": 10000000.0
  }
}

In [ ]:
# Create the query headers
request_headers: dict = {}
request_headers["Content-Type"] = "application/json"
request_headers["x-ibm-client-id"] = EAPI_CLIENT_ID
request_headers["Authorization"] = "Bearer " + jwt_token

# Submit the request
response = requests.post(EAPI_ENDPOINT, 
                         headers = request_headers,
                         data = json.dumps(payload))

For more information about allowable parameters for the payload, please see [Emissions API Developer Guide](https://developer.ibm.com/apis/catalog/ghgemissions--ibm-envizi-emissions-api/Introduction).

In [ ]:
if response.text != "":
    # Get the response as json
    response_json = response.json()
    
    # Get json and convert to dataframe
    json_str = json.dumps(response_json)
    dict = json.loads(json_str)
    dataframe = pd.json_normalize(dict) 
    
    # display
    print("\n\n")
    pd.set_option('display.max_colwidth', None)
    display( dataframe) 
else:
    print("Empty Response") 

### Output Explanation

<b>transactionId</b> - An Emissions API transaction id.

<b>totalCO2e</b>     - The total emissions as CO<sub>2</sub> equivalent (CO<sub>2</sub>e)

<b>co2</b>           - The amount of CO<sub>2</sub> (Carbon Dioxide) in the CO<sub>2</sub>e value.

<b>ch4</b>           - The amount of CH<sub>4</sub> (Methane) in the CO<sub>2</sub>e value.

<b>n2O</b>           - The amount of N<sub>2</sub>O (Nitrous Oxide) in the CO<sub>2</sub>e value.

<b>hfc</b>           - The amount of HFCs (Hydrofluorocarbons) in the CO<sub>2</sub>e value.

<b>pfc</b>           - The amount of PFCs (Perfluorocarbons) in the CO<sub>2</sub>e value.

<b>sf6</b>           - The amount of SF<sub>6</sub> (Sulphur Hexafluoride) in the CO<sub>2</sub>e value.

<b>nf3</b>           - The amount of NF<sub>3</sub> (Nitrogen Trifluoride) in the CO<sub>2</sub>e value.

<b>bioCo2</b>        - The amount of bio CO<sub>2</sub> in the CO<sub>2</sub> value.

<b>indirectCo2e</b>  - The amount of CO<sub>2</sub>e that is indirect in the CO<sub>2</sub>e value.

<b>unit</b>          - The unit of measure of the values.

<b>description</b>   - A description of the source factor set of the factor used in the calculation.

## Related Links

[Emissions API Developer Guide](https://developer.ibm.com/apis/catalog/ghgemissions--ibm-envizi-emissions-api/Introduction)